# Electronic Health Record Mock data

In [5]:
# 1. Create a simulated EHR dataset with some intentional flaws
import pandas as pd
import sqlite3
ehr_data = {
    "visit_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007],
    "patient_id": [
        "P001",
        "P002",
        "P001",
        "P003",
        "P004",
        "P002",
        "P001",
    ], 
    "patient_name": [
        "James Smith",
        "Maria Garcia",
        "James Smith",
        "Robert Chen",
        "Lisa Simpson",
        "Maria Garcia",
        "James Smith",
    ],
    "admission_date": [
        "2026-03-01",
        "2026-03-05",
        "2026-03-10",
        "2026-03-12",
        "2026-03-15",
        "2026-03-18",
        "2026-03-25",
    ],
    "discharge_date": [
        "2026-03-04",
        "2026-03-02",
        "2026-03-15",
        "2026-03-10",
        "2026-03-20",
        "2026-03-17",
        "2026-03-24",
    ],  
}

df_ehr = pd.DataFrame(ehr_data)

# 2. Connect to an in-memory SQL database
conn = sqlite3.connect(":memory:")

# 3. Push the data to a SQL table called 'ehr_visits'
df_ehr.to_sql("ehr_visits", conn, index=False, if_exists="replace")


# 4. Create our run function
def run_query(query):
    return pd.read_sql_query(query, conn)


print("Day 1 EHR Database is ready for auditing!")

Day 1 EHR Database is ready for auditing!


# Query for impossible admission dates --- data corruption

In [ ]:
# function to run our SQL queris and return a clean Pandas DataFrame
def run_audit(query):
    return pd.read_sql_query(query, conn)

# find any visits where the dischage_date happens before the admission_date.
impossible_admissionDate = """
SELECT * FROM ehr_visits
WHERE admission_date > discharge_date
"""
print('-----data corruption event----- ')
display(run_audit(impossible_admissionDate))

-----data corruption event----- 


,visit_id,patient_id,patient_name,admission_date,discharge_date
0,1002,P002,Maria Garcia,2026-03-05,2026-03-02
1,1004,P003,Robert Chen,2026-03-12,2026-03-10
2,1006,P002,Maria Garcia,2026-03-18,2026-03-17
3,1007,P001,James Smith,2026-03-25,2026-03-24


# Number of admission of each patient

In [12]:
# Query for number of admissions of each patient
query_patient_admission_numbers = """
SELECT patient_name, count(*) AS total_visits FROM ehr_visits
GROUP BY patient_name
ORDER BY total_visits DESC

"""

print('--- The number of vists of each patient are as follows----')
display(run_audit(query_patient_admission_numbers))


--- The number of vists of each patient are as follows----


,patient_name,total_visits
0,James Smith,3
1,Maria Garcia,2
2,Robert Chen,1
3,Lisa Simpson,1
